# Lab share of the individuality space (supplementary)

How much of each discriminant's between-mouse variance sits between labs. Numbers from `lab_controls.py --only ldvar` (`results/lab_controls/ldvar.csv`, `ldvar_total_null.csv`); this notebook only draws them.

| panel | what |
|---|---|
| a | each LD's share of the between-mouse variance (variance of the mouse means along it, over the sum across all LDs); line: cumulative |
| b | lab share of each LD's between-mouse variance: η² of lab on the 56 mouse means. Grey band: the same with lab labels shuffled across mice, lab sizes kept (up to its 95th percentile); dashed: its mean. Dark: p < 0.05 against the shuffles, uncorrected (a description of every LD, not a screen; at 55 LDs about 3 would pass by chance); white: n.s. |
| c | the whole space: lab share of ALL between-mouse variance (each LD's η² weighted by its share in a), against the same shuffles |

**Read with:** η² has a floor of (labs − 1) / (mice − 1) ≈ 0.18 with 11 labs and 56 mice — a random grouping of the mice "explains" that much — so bars are judged against the band, not against zero.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
for p in [str(PI), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
import paper_style as ps
import fig_layout as fl
ps.use('paper')
FIG = 'fig_lab_share'

LDVAR = pd.read_csv(PI / 'results' / 'lab_controls' / 'ldvar.csv')
TOTAL = pd.read_csv(PI / 'results' / 'lab_controls' / 'ldvar_total_null.csv')
N_SHOW = len(LDVAR)        # every discriminant; set e.g. 20 to show only the first ones
DARK, LIGHT = '#12355B', '#7FB0D8'
D = LDVAR.head(N_SHOW)
print(f"{len(LDVAR)} LDs; p < 0.05 (uncorrected): LD {list(D.loc[D['p'] < 0.05, 'LD'])}")
print(f"whole space: lab share {TOTAL['observed'].iloc[0]:.3f}, shuffled {TOTAL['weighted_null'].mean():.3f}")

## Panels

In [ ]:
def ld_axis(ax, x):
    ax.set_xlim(0.4, x[-1] + 0.6)
    ticks = [1] + list(range(5, x[-1] + 1, 5))
    ax.set_xticks(ticks)

def draw_A(fig, spec):
    ax = fig.add_subplot(spec)
    x = D['LD'].to_numpy()
    ax.bar(x, D['share_between_mouse_var'], width=0.7, color=DARK, lw=0)
    ax2 = ax.twinx()
    ax2.plot(x, LDVAR['share_between_mouse_var'].cumsum().head(N_SHOW), color=LIGHT, lw=1.0)
    ax2.set_ylim(0, 1.02); ax2.set_yticks([0, 0.5, 1]); ax2.set_ylabel('Cumulative', color=LIGHT)
    ax2.tick_params(axis='y', colors=LIGHT); ax2.spines['right'].set_visible(True); ax2.spines['right'].set_color(LIGHT)
    ax2.spines['top'].set_visible(False)
    ld_axis(ax, x); ax.tick_params(labelbottom=False)
    ax.set_ylabel('Share of between-\nmouse variance')
    return [ax, ax2]

def draw_B(fig, spec):
    ax = fig.add_subplot(spec)
    x = D['LD'].to_numpy()
    edges = np.r_[x - 0.5, x[-1] + 0.5]
    ax.fill_between(edges, 0, np.r_[D['null_95'], D['null_95'].iloc[-1]], step='post', color=ps.NULL_BAND, alpha=0.3, lw=0)
    ax.step(edges, np.r_[D['null_mean'], D['null_mean'].iloc[-1]], where='post', color='0.45', lw=0.6, ls='--')
    fill = np.where(D['p'] < 0.05, DARK, 'white')          # uncorrected: each LD is described, not screened
    ax.bar(x, D['eta2_lab'], width=0.7, color=fill, edgecolor=DARK, lw=0.5)
    ld_axis(ax, x)
    ax.set_ylim(0, 1); ax.set_yticks([0, 0.5, 1])
    ax.set_xlabel('Linear discriminant'); ax.set_ylabel('Lab share of the\nLD\'s between-mouse var.')
    h = [Patch(facecolor=DARK, edgecolor=DARK, label='p < 0.05'), Patch(facecolor='white', edgecolor=DARK, label='n.s.'),
         Patch(color=ps.NULL_BAND, alpha=0.3, lw=0, label='shuffled labs (95%)'),
         Line2D([], [], color='0.45', lw=0.6, ls='--', label='shuffled labs (mean)')]
    ax.legend(handles=h, frameon=False, loc='upper right', ncol=5, handlelength=1.0, columnspacing=0.8,
              borderaxespad=0.1)
    return [ax]

def draw_C(fig, spec):
    ax = fig.add_subplot(spec)
    nul, obs = TOTAL['weighted_null'].to_numpy(), TOTAL['observed'].iloc[0]
    ax.hist(nul, bins=np.linspace(0, 1, 201), color=ps.NULL_BAND, alpha=0.6, lw=0)
    ax.axvline(obs, color=DARK, lw=1.2)
    p = (1 + (nul >= obs).sum()) / (1 + len(nul))
    ax.text(obs + 0.03, ax.get_ylim()[1] * 0.98, f'observed\n{obs:.2f}\np {"< " if p < 1e-3 else "= "}{max(p, 1e-3):.3f}',
            ha='left', va='top', color=DARK)
    ax.text(np.mean(nul) - 0.04, ax.get_ylim()[1] * 0.5, 'shuffled\nlabs', ha='right', va='center', color='0.4')
    ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1]); ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.set_xlabel('Lab share of all\nbetween-mouse variance')
    return [ax]

fig = plt.figure(figsize=(fl.FIG_WIDTH, 3.0))
g = fig.add_gridspec(1, 2, width_ratios=[3.2, 1], wspace=0.3, left=0.125, right=0.985, top=0.92, bottom=0.17)
gl = g[0].subgridspec(2, 1, height_ratios=[0.8, 1.2], hspace=0.3)
panels = {'a': draw_A(fig, gl[0]), 'b': draw_B(fig, gl[1]), 'c': draw_C(fig, g[1])}
fl.place_letters(fig, panels, rows=['ac', 'b'], left='ab')
plt.show()
OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf', OUT / f'{FIG}.png']:
    fig.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))